In [1]:
import os

In [2]:
%pwd

'c:\\Users\\MANJU\\OneDrive\\Desktop\\MajorProjectResearchPaper\\EndToEndMajorProject\\research'

In [3]:
os.chdir("C:/Users/MANJU/OneDrive/Desktop/MajorProjectResearchPaper/EndToEndMajorProject")

In [4]:
%pwd

'C:\\Users\\MANJU\\OneDrive\\Desktop\\MajorProjectResearchPaper\\EndToEndMajorProject'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen= True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    training_data: Path
    params_epochs: int
    params_batch_size: int
    params_is_augmentation: bool
    params_image_size: list
    

In [6]:
from src.cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml,create_directories
import tensorflow as tf

[2026-04-02 00:33:26,257: WARNING: module_wrapper: From c:\Users\MANJU\Anaconda3\envs\diabetic_retinopathy\lib\site-packages\keras\src\losses.py:2976: The name tf.losses.sparse_softmax_cross_entropy is deprecated. Please use tf.compat.v1.losses.sparse_softmax_cross_entropy instead.
]


In [11]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    
    def get_training_config(self) -> TrainingConfig:
        training = self.config.training
        prepare_base_model = self.config.prepare_base_model
        params = self.params
        training_data = Path(self.config.data_transformation.transformed_data_path)
        create_directories([Path(training.root_dir)])

        training_config = TrainingConfig(
            root_dir = Path(training.root_dir),
            trained_model_path = Path(training.trained_model_path),
            updated_base_model_path = Path(prepare_base_model.updated_base_model_path),
            training_data = Path(training_data),
            params_epochs = params.EPOCHS,
            params_batch_size = params.BATCH_SIZE,
            params_is_augmentation = params.AUGMENTATION,
            params_image_size = params.IMAGE_SIZE
        )

        return training_config

In [12]:
import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf
import time

In [13]:
class Training:
    def __init__(self,config:TrainingConfig):
        self.config = config

    def get_base_model(self):
        self.model = tf.keras.models.load_model(
            self.config.updated_base_model_path
        )

    def train_valid_generator(self):

        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split = 0.20
        )

        dataflow_kwargs = dict(
            target_size = self.config.params_image_size[:-1],
            batch_size = self.config.params_batch_size,
            interpolation = "bilinear",
            class_mode = "categorical"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory =self.config.training_data,
            subset = "validation",
            shuffle = False,
            **dataflow_kwargs
        )


        if self.config.params_is_augmentation:
            train_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
                rotation_range = 40,
                horizontal_flip = True,
                width_shift_range = 0.2,
                height_shift_range = 0.2,
                shear_range = 0.2,
                zoom_range = 0.2,
                **datagenerator_kwargs 
            )
        else:
            train_datagenerator = valid_datagenerator

        self.train_generator = train_datagenerator.flow_from_directory(
            directory =self.config.training_data,
            subset = "training",
            shuffle = True,
            **dataflow_kwargs
        )

        print("Class indices:", self.train_generator.class_indices)
        print("Num classes:", self.train_generator.num_classes)

    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)

    def train(self):
        num_classes = self.train_generator.num_classes

        x = self.model.layers[-2].output
        output = tf.keras.layers.Dense(num_classes, activation='softmax')(x)

        self.model = tf.keras.Model(inputs=self.model.input, outputs=output)

        self.model.compile(
            optimizer='adam',
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )

        self.steps_per_epoch = self.train_generator.samples // self.train_generator.batch_size
        self.validation_steps = self.valid_generator.samples // self.valid_generator.batch_size

        self.model.fit(
            self.train_generator,
            epochs = self.config.params_epochs,
            steps_per_epoch = self.steps_per_epoch,
            validation_steps = self.validation_steps,
            validation_data = self.valid_generator
        )

        self.save_model(
            path = self.config.trained_model_path,
            model = self.model
        )

In [14]:
try:
    config = ConfigurationManager()
    training_config = config.get_training_config()
    training = Training(config=training_config)
    training.get_base_model()
    training.train_valid_generator()
    training.train()

except Exception as e:
    raise e 
    

[2026-04-02 00:42:09,568: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-04-02 00:42:09,601: INFO: common: yaml file: params.yaml loaded successfully]


[2026-04-02 00:42:09,620: INFO: common: created directory at:artifacts]
[2026-04-02 00:42:09,643: INFO: common: created directory at:artifacts\training]
[2026-04-02 00:42:11,176: WARNING: module_wrapper: From c:\Users\MANJU\Anaconda3\envs\diabetic_retinopathy\lib\site-packages\keras\src\backend.py:1398: The name tf.executing_eagerly_outside_functions is deprecated. Please use tf.compat.v1.executing_eagerly_outside_functions instead.
]
[2026-04-02 00:42:11,406: WARNING: module_wrapper: From c:\Users\MANJU\Anaconda3\envs\diabetic_retinopathy\lib\site-packages\keras\src\layers\pooling\max_pooling2d.py:161: The name tf.nn.max_pool is deprecated. Please use tf.nn.max_pool2d instead.
]
Found 731 images belonging to 5 classes.
Found 2931 images belonging to 5 classes.
Class indices: {'0': 0, '1': 1, '2': 2, '3': 3, '4': 4}
Num classes: 5
[2026-04-02 00:42:13,982: WARNING: module_wrapper: From c:\Users\MANJU\Anaconda3\envs\diabetic_retinopathy\lib\site-packages\keras\src\optimizers\__init__.py

c:\Users\MANJU\Anaconda3\envs\diabetic_retinopathy\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
